# 🎨 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT LAPTOP
Hệ thống tự động hóa Video Whiteboard 3 Chế Độ Lấy Ảnh & Thay Ảnh Trực Tiếp:
- 🚀 **Chế độ 1**: Kéo ảnh **Google Flow từ Laptop (thư mục `image-temp/`) qua Tunnel** ➔ Khử viền đen & Vector hóa ➔ **Tự động lưu vào Google Drive `/image/f/gen/`** ➔ Đóng gói VideoScribe!
- 📂 **Chế độ 2**: Tìm và bốc ảnh có sẵn trong kho Google Drive (`/image/f/`).
- 🎨 **Chế độ 3**: Tự động sinh ảnh AI Doodle 100% (Pollinations + vtracer).
- 📤 **Thay ảnh trực tiếp**: Tải file ảnh PNG/JPG/SVG từ máy lên thay thế trực tiếp cho bất kỳ cảnh nào!

## ⚡ BƯỚC 1: Cài Đặt Môi Trường & Kết Nối Google Drive (Chạy 1 lần)

In [ ]:
from google.colab import drive
import os
import sys

print("🔗 Đang yêu cầu quyền truy cập Google Drive...")
drive.mount('/content/drive')

print("⏳ Đang cài đặt thư viện lõi (Whisper, Gemini, vtracer, Gradio, Pillow, numpy, ffmpeg)...")
!apt-get install -y ffmpeg
!pip install -q openai-whisper google-genai requests vtracer Pillow numpy gradio

print("✅ Đã cài đặt xong toàn bộ môi trường! Hãy chuyển sang BƯỚC 2 để mở Giao Diện.")

## 🎛️ BƯỚC 2: Khởi Chạy Giao Diện Web Điều Khiển Toàn Diện (All-In-One UI)

In [ ]:
import os
import re
import glob
import json
import time
import random
import shutil
import zipfile
import subprocess
import urllib.request
import urllib.parse
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import vtracer
import whisper
import requests
import gradio as gr

ASSETS_DIR = "assets"
os.makedirs(ASSETS_DIR, exist_ok=True)

STOP_WORDS = {
    "outline", "sketch", "vector", "line", "art", "drawing", "illustration", 
    "clipart", "transparent", "icon", "svg", "png", "jpg", "the", "a", "an", 
    "of", "in", "on", "with", "and", "or", "for", "to", "black", "white", 
    "hollow", "simple", "doodle", "style", "clean", "minimal", "graphic", "concept"
}

def clean_slug(text):
    text = re.sub(r'[^a-zA-Z0-9\s_-]', '', text)
    text = re.sub(r'\s+', '_', text).strip('_').lower()
    return text[:40] if text else "doodle_icon"

def videoscribe_escape(s):
    s = s.replace('&', '&amp;')
    s = s.replace('<', '&lt;')
    s = s.replace('"', '&quot;')
    return s

def clean_and_devignette_image(input_img_path, output_png_path, threshold=205):
    """Tẩy trắng nền 100%, khử sạch bóng mờ ở 4 góc và triệt tiêu khung viền đen mép ảnh."""
    with Image.open(input_img_path) as img:
        img = img.convert('RGB')
        arr = np.array(img, dtype=np.uint8)
        gray = np.mean(arr, axis=2)
        arr[gray > threshold] = [255, 255, 255]
        
        pad = 8
        arr[:pad, :] = [255, 255, 255]
        arr[-pad:, :] = [255, 255, 255]
        arr[:, :pad] = [255, 255, 255]
        arr[:, -pad:] = [255, 255, 255]
        
        cleaned = Image.fromarray(arr)
        cleaned.save(output_png_path, 'PNG')
    return output_png_path

def ensure_svg_file(input_file_path, output_svg_path, color_mode="binary"):
    """Đảm bảo file đầu ra luôn là 1 file SVG vector mượt mà, khử sạch loang lổ và viền đen."""
    os.makedirs(os.path.dirname(os.path.abspath(output_svg_path)), exist_ok=True)
    
    is_bitmap = False
    if os.path.exists(input_file_path):
        try:
            with open(input_file_path, 'rb') as check_f:
                header = check_f.read(8)
                if header.startswith(b'\x89PNG') or header.startswith(b'\xff\xd8') or header.startswith(b'GIF8') or header.startswith(b'RIFF'):
                    is_bitmap = True
        except Exception: pass

    if is_bitmap or input_file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
        temp_cleaned_png = output_svg_path + ".cleaned.png"
        try:
            clean_and_devignette_image(input_file_path, temp_cleaned_png, threshold=205)
            
            if color_mode == "color":
                vtracer.convert_image_to_svg_py(
                    temp_cleaned_png,
                    output_svg_path,
                    colormode='color',
                    hierarchical='stacked',
                    mode='spline',
                    filter_speckle=8,
                    color_precision=6,
                    layer_difference=16,
                    corner_threshold=60,
                    length_threshold=4.0,
                    max_iterations=10,
                    splice_threshold=45,
                    path_precision=3
                )
            else:
                vtracer.convert_image_to_svg_py(
                    temp_cleaned_png,
                    output_svg_path,
                    colormode='binary',
                    hierarchical='stacked',
                    mode='spline',
                    filter_speckle=10,
                    corner_threshold=70,
                    length_threshold=5.0,
                    max_iterations=10,
                    splice_threshold=50,
                    path_precision=3
                )
            if os.path.exists(temp_cleaned_png): os.remove(temp_cleaned_png)
            return output_svg_path
        except Exception:
            if os.path.exists(temp_cleaned_png): os.remove(temp_cleaned_png)
            
    if os.path.exists(input_file_path):
        if os.path.abspath(input_file_path) != os.path.abspath(output_svg_path):
            shutil.copy(input_file_path, output_svg_path)
    else:
        with open(output_svg_path, "w", encoding="utf-8") as f:
            f.write('<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">Doodle</text></svg>')
    return output_svg_path

def build_drive_cache(drive_search_dirs):
    cache = []
    for s_dir in drive_search_dirs:
        if os.path.exists(s_dir):
            for root, dirs, files in os.walk(s_dir):
                for f in files:
                    if f.lower().endswith('.svg') or f.lower().endswith('.png') or f.lower().endswith('.jpg'):
                        clean_n = re.sub(r'[^a-zA-Z0-9]', ' ', os.path.splitext(f)[0]).lower()
                        file_words = set([w for w in clean_n.split() if w not in STOP_WORDS and len(w) > 2])
                        cache.append({
                            "path": os.path.join(root, f),
                            "filename": f,
                            "words": file_words,
                            "clean_name": clean_n
                        })
    return cache

def generate_doodle_svg(prompt_keyword, target_svg_path, drive_save_path=None):
    os.makedirs(os.path.dirname(os.path.abspath(target_svg_path)), exist_ok=True)
    ai_prompt = (
        f"clean minimalist black and white whiteboard line art icon of {prompt_keyword}, "
        f"crisp black outline drawing, pure 100% white background, hollow shapes, fine sharp pen strokes, coloring book style, no background shadow, no frame, no border, no shading"
    )
    encoded_prompt = urllib.parse.quote(ai_prompt)
    seed = random.randint(1000, 999999)
    url = f"https://image.pollinations.ai/prompt/{encoded_prompt}?width=768&height=768&model=flux&nologo=true&seed={seed}"
    
    temp_img = target_svg_path + ".temp_dl"
    temp_clean = target_svg_path + ".temp_clean.png"
    success = False
    
    for attempt in range(3):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=25) as resp:
                with open(temp_img, 'wb') as f:
                    f.write(resp.read())
            
            clean_and_devignette_image(temp_img, temp_clean, threshold=205)
            
            vtracer.convert_image_to_svg_py(
                temp_clean,
                target_svg_path,
                colormode='binary',
                hierarchical='stacked',
                mode='spline',
                filter_speckle=10,
                corner_threshold=70,
                length_threshold=5.0,
                max_iterations=10,
                splice_threshold=50,
                path_precision=3
            )
            success = True
            break
        except Exception:
            time.sleep(1.5)
        finally:
            if os.path.exists(temp_img): os.remove(temp_img)
            if os.path.exists(temp_clean): os.remove(temp_clean)
            
    if success and os.path.exists(target_svg_path) and os.path.getsize(target_svg_path) > 100:
        if drive_save_path:
            try:
                os.makedirs(os.path.dirname(os.path.abspath(drive_save_path)), exist_ok=True)
                if os.path.abspath(target_svg_path) != os.path.abspath(drive_save_path):
                    shutil.copy(target_svg_path, drive_save_path)
            except Exception: pass
        return target_svg_path
    else:
        with open(target_svg_path, "w", encoding="utf-8") as f:
            f.write(f'<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"><rect width="500" height="500" fill="none" stroke="#000" stroke-width="4"/><text x="250" y="250" font-size="26" text-anchor="middle" fill="#000">{prompt_keyword}</text></svg>')
        return target_svg_path

def build_scribe_file(audio_path, metadata_path="scene_metadata.json"):
    if not os.path.exists(metadata_path): return None
    with open(metadata_path, "r", encoding="utf-8") as f:
        meta_data = json.load(f)

    BUILD_DIR = "scribe_build"
    if os.path.exists(BUILD_DIR): shutil.rmtree(BUILD_DIR)
    os.makedirs(BUILD_DIR, exist_ok=True)

    target_audio = os.path.join(BUILD_DIR, "voiceover.mp3")
    has_audio = False
    
    preferred_audio = None
    if audio_path and os.path.exists(audio_path):
        preferred_audio = audio_path
    elif os.path.exists("voiceover.mp3"):
        preferred_audio = "voiceover.mp3"
    else:
        mp3s = glob.glob("*.mp3")
        if mp3s:
            preferred_audio = next((f for f in mp3s if "voice" in f.lower()), mp3s[0])

    if preferred_audio:
        try:
            cmd = f'ffmpeg -y -i "{preferred_audio}" -ar 44100 -ac 1 -b:a 128k "{target_audio}"'
            res = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            if res.returncode == 0 and os.path.exists(target_audio):
                has_audio = True
            else:
                shutil.copy(preferred_audio, target_audio)
                has_audio = True
        except Exception:
            shutil.copy(preferred_audio, target_audio)
            has_audio = True

    drawing_xml = os.path.join(BUILD_DIR, "drawing.xml")
    audio_attr = 'voiceOver="voiceover.mp3" voiceOverVolume="1"' if has_audio else ''
    options_val = '&lt;drawingOptions paperStyle=&quot;1&quot; paperColour=&quot;16777215&quot; threeDMode=&quot;no&quot; loopSound=&quot;no&quot; zoomAtEnd=&quot;no&quot; vignette=&quot;0&quot; xPerspective=&quot;0&quot; yPerspective=&quot;0&quot; zPerspective=&quot;0&quot;/>'
    
    unique_proj_id = str(random.randint(1000000000, 9999999999))
    proj_name = f"Auto_Project_{int(time.time())}"
    
    with open(drawing_xml, "w", encoding="utf-8") as f:
        f.write('<?xml version="1.0" encoding="utf-8"?>\n')
        f.write(f'<drawing app="VideoScribe" ver="3.7.3103" filever="5" name="{proj_name}" desc="" tags="" uniqueID="{unique_proj_id}" isDescendedFromTemplate="not_desc" defaultHandMD5="default_right" options="{options_val}" backingTrack="" lastRenderDateTime="Invalid Date" {audio_attr}>\n')
        
        xml_elements = []
        element_counter = 1000000000 + random.randint(10000, 99999)
        visual_timeline_ms = 0
        
        for scene_idx, scene in enumerate(meta_data):
            raw_images = scene.get('images', [])
            n = len(raw_images)
            if n == 0: continue
            
            if scene_idx < len(meta_data) - 1:
                speech_dur_s = float(meta_data[scene_idx + 1].get('start', 0)) - float(scene.get('start', 0))
            else:
                speech_dur_s = float(scene.get('end', 0)) - float(scene.get('start', 0))
            if speech_dur_s <= 0: speech_dur_s = (float(scene.get('end', 0)) - float(scene.get('start', 0))) or (3.5 * n)
            duration_per_img = speech_dur_s / n
            scene_x = scene_idx * 1600
            scene_y = 0
            cam_scale = 0.82
            cam_x = 448.5 - scene_x * cam_scale
            cam_y = 252.5 - scene_y * cam_scale
            
            for i, img_meta in enumerate(raw_images):
                filename = img_meta.get('file_name', '')
                file_path = os.path.join(ASSETS_DIR, filename)
                
                if not os.path.exists(file_path):
                    alt_svg = os.path.splitext(file_path)[0] + ".svg"
                    if os.path.exists(alt_svg): file_path = alt_svg
                    else: continue
                
                ensure_svg_file(file_path, file_path)
                actual_filename = os.path.basename(file_path)

                content = ""
                try:
                    with open(file_path, "r", encoding="utf-8", errors="replace") as f2:
                        raw_svg = f2.read()
                        raw_svg = re.sub(r'<\?xml[^>]*\?>', '', raw_svg)
                        raw_svg = re.sub(r'<!DOCTYPE[^>]*>', '', raw_svg)
                        raw_svg = re.sub(r'<!--.*?-->', '', raw_svg, flags=re.DOTALL)
                        content = raw_svg.replace('\n', ' ').replace('\r', '')
                except Exception:
                    content = '<svg xmlns="http://www.w3.org/2000/svg" width="500" height="500"></svg>'
                    
                element_counter += random.randint(1000, 5000)
                if n == 1: pos_x, pos_y, scale_val = scene_x, scene_y, "0.8"
                elif n == 2: pos_x, pos_y, scale_val = scene_x + (-250 if i == 0 else 250), scene_y, "0.55"
                else: pos_x, pos_y, scale_val = scene_x + (-220 if i == 1 else (220 if i == 2 else 0)), scene_y + (120 if i > 0 else -120), "0.45"
                    
                ai_style = img_meta.get('animation_style', 'draw')
                if ai_style == 'draw': draw_style = 'draw_style_normal'
                elif ai_style in ['movein', 'movein_hand', 'movein_nohand']: draw_style = 'draw_style_movein'
                elif ai_style == 'fadein': draw_style = 'draw_style_fadein'
                else: draw_style = 'draw_style_normal'
                
                movin_compass = str(random.randint(1, 8))
                draw_detail = 'yes' if draw_style == 'draw_style_normal' else 'no'
                custom_hand = 'default_nohand' if draw_style == 'draw_style_movein' else ''
                movin_arc = random.choice(['0', '1']) if draw_style == 'draw_style_movein' else '0'
                
                total_time_ms = int(duration_per_img * 1000)
                # Cài đặt chuẩn: Chuyển cảnh 0.5s (500ms), Dừng hình 0.5s - 1.0s, Bàn tay vẽ chuẩn xác 100% không lệch
                if total_time_ms <= 1500:
                    trans_time_ms = max(200, int(total_time_ms * 0.2))
                    pause_time_ms = max(300, int(total_time_ms * 0.25))
                    target_time_ms = max(200, total_time_ms - trans_time_ms - pause_time_ms)
                elif total_time_ms <= 2500:
                    trans_time_ms = 500
                    pause_time_ms = 500
                    target_time_ms = max(200, total_time_ms - trans_time_ms - pause_time_ms)
                else:
                    trans_time_ms = 500
                    pause_time_ms = min(1000, max(500, int(total_time_ms * 0.25)))
                    target_time_ms = max(200, total_time_ms - trans_time_ms - pause_time_ms)
                
                drawing_xml_attr = f'drawingXML="{videoscribe_escape(content)}"'
                
                element_xml = (
                    f'  <element elementType="drawing" descName="" elementID="{element_counter}" '
                    f'splitTextField="no" drawingText="" fontName="null" {drawing_xml_attr} '
                    f'customHandMD5="{custom_hand}" colourEffect="0" targetTime="{target_time_ms}" '
                    f'pauseTime="{pause_time_ms}" transitionTime="{trans_time_ms}" '
                    f'drawStyle="{draw_style}" rotation="0" visible="true" '
                    f'currentPosX="{pos_x}" currentPosY="{pos_y}" offsetX="{pos_x}" offsetY="{pos_y}" '
                    f'scalesX="0.8" scalesY="0.8" theScale="0.8" targetHeight="800" '
                    f'movinCompass="{movin_compass}" movinFlow="0" movinArc="{movin_arc}" movinAllowRotate="yes" '
                    f'drawDetail="{draw_detail}" sketchStyle="no" brush="0" brushOptions="0" opacity="1" '
                    f'textColour="-1" textAlign="left" textBackwards="no" rtlLanguage="no" textSpacing="0" '
                    f'flipHoriz="no" flipVert="no" locked="no" calligraphy_angle="45" keepRunning="no" '
                    f'loopOptions="Fit to Time" blendMode="normal" filters="&lt;filters/>" morphFromID="0" '
                    f'morphCamera="no" morphRemoveOld="yes" cameraPositionX="{cam_x}" cameraPositionY="{cam_y}" '
                    f'cameraScale="{cam_scale}" cameraCanvasWid="897.7777777777778" cameraCanvasHei="505" '
                    f'availableRecolours="&lt;availableRecolours/>" recolouringSchemes="&lt;recolouringSchemes/>" '
                    f'skinTone="-1" hairColour="-1" highlightColour="-1" customColour1="-1" customColour2="-1" '
                    f'originalOutlineColour="0" greyscaleContrast="70" />\n'
                )
                xml_elements.append(element_xml)
                visual_timeline_ms += total_time_ms
                
        f.write('\n'.join(xml_elements) + '\n')
        f.write('</drawing>')

    time_str = time.strftime("%Y%m%d_%H%M%S")
    out_file = f"Auto_Project_{time_str}.scribe"
    
    with zipfile.ZipFile(out_file, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for root, dirs, files in os.walk(BUILD_DIR):
            for file in files:
                f_p = os.path.join(root, file)
                zipf.write(f_p, os.path.relpath(f_p, BUILD_DIR))

    tmp_p = out_file + ".tmp"
    with zipfile.ZipFile(out_file, 'r') as zin, zipfile.ZipFile(tmp_p, 'w', zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = zin.read(item.filename)
            if item.filename == 'drawing.xml':
                data = data.decode('utf-8', errors="replace").replace('&gt;', '>').encode('utf-8')
            zout.writestr(item, data)
    os.replace(tmp_p, out_file)
    return out_file

def generate_preview_cards(meta_data=None):
    if meta_data is None:
        if not os.path.exists("scene_metadata.json"): return "<p style='padding:20px; color:#718096;'>Chưa có dữ liệu kịch bản.</p>"
        with open("scene_metadata.json", "r", encoding="utf-8") as f:
            meta_data = json.load(f)
            
    cards_html = ""
    for s in meta_data:
        sc_id = s.get('sentence_id', 1)
        start_t = float(s.get('start', 0))
        end_t = float(s.get('end', 0))
        dur_t = end_t - start_t
        speech = s.get('speech_text', s.get('text', ''))
        
        imgs_div = ""
        for img_idx, img in enumerate(s.get('images', [])):
            fp = os.path.join(ASSETS_DIR, img.get('file_name', ''))
            svg_content = ""
            if os.path.exists(fp):
                try:
                    with open(fp, "r", encoding="utf-8", errors="replace") as svg_f:
                        raw_c = svg_f.read()
                        if "<svg" in raw_c:
                            svg_content = re.sub(r'<\?xml[^>]*\?>', '', raw_c)
                        else:
                            svg_content = '<div style="color:#718096; font-size:12px;">[Ảnh Vector]</div>'
                except Exception:
                    svg_content = '<div style="color:#718096; font-size:12px;">[Ảnh]</div>'
                    
            imgs_div += f'''
            <div style="background:#fff; border:1px solid #cbd5e0; border-radius:10px; padding:10px; margin:6px; display:inline-block; vertical-align:top; width:160px; text-align:center; box-shadow:0 2px 4px rgba(0,0,0,0.05);">
                <div style="font-size:11px; font-weight:bold; color:#718096; margin-bottom:4px;">Ảnh #{img_idx+1} (Cảnh {sc_id})</div>
                <div style="height:110px; display:flex; align-items:center; justify-content:center; overflow:hidden; background:#f8fafc; border-radius:6px; padding:4px;">
                    {svg_content if svg_content else '<div style="color:#a0aec0;">Chưa có ảnh</div>'}
                </div>
                <div style="font-size:12px; font-weight:bold; color:#1a202c; margin-top:8px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;" title="{img.get('svg_search_prompt','')}">
                    {img.get('svg_search_prompt','')}
                </div>
                <div style="font-size:11px; color:#4a5568; margin-top:2px;">🎭 {img.get('animation_style','draw').upper()}</div>
                <div style="font-size:10px; color:#2b6cb0; margin-top:3px; background:#ebf8ff; padding:2px 4px; border-radius:4px; overflow:hidden; text-overflow:ellipsis; white-space:nowrap;">
                    {img.get('source','')}
                </div>
            </div>
            '''

        cards_html += f'''
        <div style="background:#f7fafc; border:1px solid #e2e8f0; border-radius:12px; padding:16px; margin-bottom:16px; box-shadow:0 1px 3px rgba(0,0,0,0.05);">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:10px; border-bottom:1px solid #e2e8f0; padding-bottom:8px;">
                <span style="font-size:16px; font-weight:bold; color:#2b6cb0;">Cảnh #{sc_id}</span>
                <span style="font-size:13px; background:#edf2f7; color:#4a5568; padding:3px 8px; border-radius:12px; font-weight:600;">⏱️ {start_t:.1f}s ➔ {end_t:.1f}s ({dur_t:.1f}s)</span>
            </div>
            <div style="font-size:14px; color:#2d3748; line-height:1.5; margin-bottom:12px; font-style:italic;">
                🗣️ "{speech}"
            </div>
            <div style="display:flex; flex-wrap:wrap; gap:8px;">
                {imgs_div}
            </div>
        </div>
        '''

    return f'''
    <div style="font-family:-apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; max-height:550px; overflow-y:auto; padding:10px;">
        {cards_html}
    </div>
    '''

# --- HÀM TỰ ĐỘNG CẮT CÂU THÔNG MINH (TỐI ƯU NHỊP ẢNH WHITEBOARD ~3.0s - 3.5s/CẢNH) ---
def split_smart_scenes(raw_segments, target_dur=3.5, max_dur=4.5):
    all_splits = []
    for item in raw_segments:
        start = float(item.get("start", 0.0))
        end = float(item.get("end", 0.0))
        text = str(item.get("text", "")).strip()
        dur = end - start
        
        if dur <= max_dur and len(text.split()) <= 8:
            clean_t = re.sub(r'^[,\.;:\—\s]+', '', text).strip()
            if clean_t:
                all_splits.append({"start": start, "end": end, "text": clean_t})
            continue
            
        sentences = re.split(r'(?<=[.!?])\s+', text)
        sub_chunks = []
        for s in sentences:
            s = s.strip()
            if not s: continue
            if len(s.split()) >= 6:
                p_parts = [p.strip() for p in re.split(r'([,;:\—]+)', s) if p.strip()]
                merged = []
                temp = ""
                for p in p_parts:
                    if p in [",", ";", ":", "—"]:
                        temp += p
                        if len(temp.split()) >= 4:
                            merged.append(temp.strip())
                            temp = ""
                    else:
                        temp = (temp + " " + p).strip() if temp else p
                        if len(temp.split()) >= 6:
                            merged.append(temp.strip())
                            temp = ""
                if temp:
                    merged.append(temp.strip())
                sub_chunks.extend(merged)
            else:
                sub_chunks.append(s)
                
        if len(sub_chunks) <= 1:
            words = text.split()
            n_chunks = max(2, int(round(dur / target_dur)))
            chunk_size = max(1, len(words) // n_chunks)
            sub_chunks = []
            for k in range(0, len(words), chunk_size):
                chunk_str = ' '.join(words[k:k+chunk_size])
                if chunk_str: sub_chunks.append(chunk_str)
                
        cleaned_chunks = []
        for c in sub_chunks:
            c = c.strip()
            if not c: continue
            if cleaned_chunks and (len(c.split()) <= 2 or len(cleaned_chunks[-1].split()) <= 2):
                cleaned_chunks[-1] = cleaned_chunks[-1] + " " + c
            else:
                cleaned_chunks.append(c)
                
        sub_chunks = cleaned_chunks if cleaned_chunks else sub_chunks
        
        total_chars = sum(len(c) for c in sub_chunks)
        cur_t = start
        for c in sub_chunks:
            ratio = len(c) / total_chars if total_chars > 0 else 1.0 / len(sub_chunks)
            c_dur = dur * ratio
            c_end = cur_t + c_dur
            clean_t = re.sub(r'^[,\.;:\—\s]+', '', c).strip()
            if clean_t:
                all_splits.append({
                    "start": round(cur_t, 2),
                    "end": round(c_end, 2),
                    "text": clean_t
                })
            cur_t = c_end

    final_scenes = []
    for idx, s in enumerate(all_splits):
        final_scenes.append({
            "sentence_id": idx + 1,
            "start": s["start"],
            "end": s["end"],
            "text": s["text"]
        })
    return final_scenes

# --- BƯỚC 1: BÓC TÁCH WHISPER & GỬI VỀ LAPTOP ---
def step1_whisper_and_send_to_laptop(audio_file_obj, bridge_url_input):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else None)
    if not audio_path and os.path.exists("voiceover.mp3"): audio_path = "voiceover.mp3"
    if not audio_path or not os.path.exists(audio_path):
        yield fmt_log("❌ Lỗi: Vui lòng chọn file âm thanh voiceover.mp3!")
        return

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL (Cloudflare Tunnel) từ Laptop!")
        return

    if os.path.exists(ASSETS_DIR):
        for f in os.listdir(ASSETS_DIR):
            try: os.remove(os.path.join(ASSETS_DIR, f))
            except Exception: pass

    yield fmt_log(f"🎙️ Âm thanh: {os.path.basename(audio_path)}")
    yield fmt_log("⏳ Whisper đang bóc tách câu thoại...")
    whisper_model = whisper.load_model("base")
    whisper_res = whisper_model.transcribe(audio_path)
    raw_segments = [{"start": seg["start"], "end": seg["end"], "text": seg["text"].strip()} for seg in whisper_res["segments"] if seg["text"].strip()]
    scenes = split_smart_scenes(raw_segments, target_dur=3.5, max_dur=4.5)
    yield fmt_log(f"✅ Đã bóc tách & tối ưu thành {len(scenes)} cảnh chuẩn nhịp Whiteboard (~3.0s - 3.5s/cảnh)!")

    yield fmt_log(f"🌐 Đang gửi {len(scenes)} câu thoại về Laptop qua {bridge_url}...")
    try:
        resp = requests.post(f"{bridge_url}/", json={"scenes": scenes}, timeout=30)
        if resp.status_code == 200:
            yield fmt_log("═"*60)
            yield fmt_log(f"🎉 ĐÃ GỬI THÀNH CÔNG VỀ LAPTOP (File: pending_scenes.json)!")
            yield fmt_log("👉 BÂY GIỜ BẠN HÃY BẢO AI TRONG IDE LAPTOP: 'Hãy phân tích kịch bản theo Prompt 1 trong prompt.md'.")
            yield fmt_log("👉 Sau khi AI phân tích xong, bạn bấm '📥 2. Kéo Kịch Bản & Tải/Vẽ Ảnh' bên dưới!")
            yield fmt_log("═"*60)
        else:
            yield fmt_log(f"⚠️ Laptop trả về mã lỗi: {resp.status_code}")
    except Exception as e:
        yield fmt_log(f"❌ Lỗi gửi về Laptop: {e}")


# --- BƯỚC 2: KÉO KỊCH BẢN TỪ LAPTOP VỀ & TẢI/VẼ ẢNH (3 CHẾ ĐỘ) ---
def step2_pull_from_laptop_and_preview(bridge_url_input, image_mode, drive_f_path, drive_gen_path):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    bridge_url = bridge_url_input.strip().rstrip('/') if bridge_url_input else ""
    if not bridge_url:
        yield fmt_log("❌ Lỗi: Vui lòng dán Local Bridge URL!"), None
        return

    yield fmt_log(f"📥 Đang kéo kịch bản từ Laptop ({bridge_url}/get_analyzed_scenes)..."), None
    try:
        resp = requests.get(f"{bridge_url}/get_analyzed_scenes", timeout=30)
        if resp.status_code != 200 or resp.json().get("status") != "success":
            yield fmt_log("⚠️ Chưa có kịch bản trên Laptop! Hãy bảo AI trong IDE phân tích file pending_scenes.json trước!"), None
            return
        raw_analyzed_data = resp.json().get("data", [])
        yield fmt_log(f"🎉 Nhận thành công {len(raw_analyzed_data)} cảnh kịch bản từ Laptop!"), None
    except Exception as e:
        yield fmt_log(f"❌ Lỗi kết nối tới Laptop: {e}"), None
        return

    os.makedirs(drive_gen_path, exist_ok=True)
    scene_metadata = []
    total_scenes = len(raw_analyzed_data)

    # --- CHẾ ĐỘ 1: KÉO ẢNH GOOGLE FLOW TỪ LAPTOP (image-temp/) QUA TUNNEL ---
    is_flow_laptop_mode = ("Laptop" in image_mode or "Tunnel" in image_mode or "Flow" in image_mode or "1." in image_mode)
    is_drive_mode = ("2." in image_mode or ("Drive" in image_mode and "Tìm" in image_mode))

    if is_flow_laptop_mode:
        yield fmt_log("🚀 Chế độ: KÉO ẢNH GOOGLE FLOW TỪ LAPTOP (image-temp/) QUA TUNNEL..."), None
        try:
            flow_resp = requests.get(f"{bridge_url}/get_flow_images", timeout=30)
            flow_images = flow_resp.json().get("images", []) if flow_resp.status_code == 200 else []
            yield fmt_log(f"📁 Tìm thấy {len(flow_images)} ảnh Flow AI trong thư mục image-temp trên Laptop!"), None
        except Exception as e:
            flow_images = []
            yield fmt_log(f"⚠️ Không lấy được danh sách ảnh Flow AI từ Laptop: {e}"), None

        flat_img_counter = 0
        for j, s in enumerate(raw_analyzed_data):
            sc_id = s.get("sentence_id", j + 1)
            raw_imgs = s.get("images", [])
            sentence_entry = {
                "sentence_id": sc_id,
                "start": float(s.get('start', 0)),
                "end": float(s.get('end', 0)),
                "speech_text": s.get('speech_text', s.get('text', '')),
                "images": []
            }
            
            for img_idx, img_info in enumerate(raw_imgs):
                file_base = f"sentence_{sc_id:03d}_img_{img_idx+1:02d}"
                kw = img_info.get("svg_search_prompt", "concept")
                target_svg = os.path.join(ASSETS_DIR, f"{file_base}.svg")
                slug_n = clean_slug(kw)
                drive_save = os.path.join(drive_gen_path, f"{slug_n}.svg")
                
                # Nếu có ảnh Flow AI tương ứng từ Laptop
                if flat_img_counter < len(flow_images):
                    flow_item = flow_images[flat_img_counter]
                    dl_url = f"{bridge_url}{flow_item['download_url']}"
                    temp_dl_png = os.path.join(ASSETS_DIR, f"{file_base}_flow.png")
                    
                    try:
                        r_img = requests.get(dl_url, timeout=30)
                        with open(temp_dl_png, 'wb') as f_out: f_out.write(r_img.content)
                        ensure_svg_file(temp_dl_png, target_svg)
                        
                        # Lưu bản SVG vector vào Google Drive /MyDrive/image/f/gen/
                        if os.path.exists(target_svg):
                            shutil.copy(target_svg, drive_save)
                        src_note = f"🚀 Flow AI: {flow_item['filename']} ➔ Drive: {os.path.basename(drive_save)}"
                    except Exception as e:
                        generate_doodle_svg(kw, target_svg, drive_save)
                        src_note = f"⚠️ Lỗi Flow AI ➔ Vẽ AI Thay Thế"
                else:
                    # Nếu thiếu ảnh Flow -> Tự vẽ AI bù vào
                    generate_doodle_svg(kw, target_svg, drive_save)
                    src_note = f"✨ Vẽ AI Mới (Thiếu Flow #{flat_img_counter+1})"
                    
                flat_img_counter += 1
                sentence_entry["images"].append({
                    "img_idx": img_idx + 1,
                    "visual_concept": img_info.get("visual_concept", ""),
                    "svg_search_prompt": kw,
                    "animation_style": img_info.get("animation_style", "draw"),
                    "file_name": os.path.basename(target_svg),
                    "source": src_note
                })
                
            scene_metadata.append(sentence_entry)
            yield fmt_log(f"   [Cảnh {sc_id}/{total_scenes}] ➔ {len(raw_imgs)} ảnh ({src_note})"), None

    # --- CHẾ ĐỘ 2 & 3: TÌM DRIVE HOẶC VẼ AI MỚI ---
    else:
        drive_cache = []
        if is_drive_mode:
            yield fmt_log(f"📂 Đang quét kho ảnh trên Google Drive ({drive_f_path})..."), None
            drive_cache = build_drive_cache([drive_f_path])
            yield fmt_log(f"✅ Tìm thấy {len(drive_cache)} ảnh có sẵn trong Drive!"), None
        else:
            yield fmt_log("🎨 Chế độ: 100% AI VẼ NÉT MỚI HOÀN TOÀN (Bộ lọc khử viền & loang lổ)"), None

        used_files = set()
        mode_flag = "drive_first" if is_drive_mode else "ai_first"

        for j, s in enumerate(raw_analyzed_data):
            sc_id = s.get("sentence_id", j + 1)
            raw_imgs = s.get("images", [])
            sentence_entry = {
                "sentence_id": sc_id,
                "start": float(s.get('start', 0)),
                "end": float(s.get('end', 0)),
                "speech_text": s.get('speech_text', s.get('text', '')),
                "images": []
            }
            
            for img_idx, img_info in enumerate(raw_imgs):
                file_base = f"sentence_{sc_id:03d}_img_{img_idx+1:02d}"
                kw = img_info.get("svg_search_prompt", "concept")
                target_p = os.path.join(ASSETS_DIR, f"{file_base}.svg")
                
                # Tìm Drive hoặc Vẽ AI
                raw_words = re.sub(r'[^a-zA-Z0-9]', ' ', kw).lower().split()
                core_query_words = set([w for w in raw_words if w not in STOP_WORDS and len(w) > 2])
                
                matched = False
                if mode_flag == "drive_first" and drive_cache and core_query_words:
                    best_matches = []
                    max_score = 0
                    for item in drive_cache:
                        overlap = core_query_words.intersection(item["words"])
                        score = len(overlap)
                        if score > 0:
                            if " ".join(core_query_words) in item["clean_name"]: score += 2.0
                            if score > max_score: max_score = score; best_matches = [item["path"]]
                            elif score == max_score: best_matches.append(item["path"])
                    if best_matches and max_score >= 1.0:
                        unused = [m for m in best_matches if m not in used_files]
                        chosen = random.choice(unused) if unused else random.choice(best_matches)
                        used_files.add(chosen)
                        ensure_svg_file(chosen, target_p)
                        src_note = f"📂 Drive: {os.path.basename(chosen)}"
                        matched = True
                        
                if not matched:
                    slug_n = clean_slug(kw)
                    drive_save = os.path.join(drive_gen_path, f"{slug_n}.svg")
                    generate_doodle_svg(kw, target_p, drive_save)
                    src_note = f"✨ AI Vẽ Mới: {os.path.basename(drive_save)}"
                
                sentence_entry["images"].append({
                    "img_idx": img_idx + 1,
                    "visual_concept": img_info.get("visual_concept", ""),
                    "svg_search_prompt": kw,
                    "animation_style": img_info.get("animation_style", "draw"),
                    "file_name": os.path.basename(target_p),
                    "source": src_note
                })
                
            scene_metadata.append(sentence_entry)
            yield fmt_log(f"   [Cảnh {sc_id}/{total_scenes}] ➔ {len(raw_imgs)} ảnh ({src_note})"), None

    with open("scene_metadata.json", "w", encoding="utf-8") as f:
        json.dump(scene_metadata, f, ensure_ascii=False, indent=2)

    yield fmt_log("═"*60), generate_preview_cards(scene_metadata)
    yield fmt_log("✨ ĐÃ TẢI & VẼ TOÀN BỘ ẢNH XONG! Hãy xem trước ảnh bên dưới. Khi đã ưng ý ➔ Bấm nút '📦 BƯỚC 3: ĐÓNG GÓI & TẢI FILE VIDEOSCRIBE'!"), generate_preview_cards(scene_metadata)

# --- BƯỚC 3: ĐÓNG GÓI VÀ TẢI FILE SCRIBE ---
def step3_build_and_download_scribe(audio_file_obj):
    logs = []
    def fmt_log(msg):
        logs.append(f"[{time.strftime('%H:%M:%S')}] {msg}")
        return "\n".join(logs)

    if not os.path.exists("scene_metadata.json"):
        yield fmt_log("❌ Chưa có dữ liệu kịch bản! Hãy chạy Bước 2 trước."), None
        return

    audio_path = audio_file_obj if isinstance(audio_file_obj, str) else (audio_file_obj.name if audio_file_obj else None)
    yield fmt_log("📦 Đang đóng gói dự án VideoScribe mới (Unique Project ID)..."), None
    out_file = build_scribe_file(audio_path, "scene_metadata.json")
    yield fmt_log(f"🎉 HOÀN TẤT! File dự án mới ({out_file}) đã sẵn sàng để tải về ở khung bên phải."), out_file

# --- VẼ LẠI ẢNH BẰNG AI CHO CẢNH ĐƯỢC CHỌN ---
def redraw_single_scene(scene_id, custom_kw, drive_gen_path):
    if not os.path.exists("scene_metadata.json"): return "Chưa có dữ liệu kịch bản!", None
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    found = False
    for s in meta:
        if s.get('sentence_id') == int(scene_id):
            found = True
            for img_idx, img in enumerate(s.get('images', [])):
                kw = custom_kw.strip() if custom_kw.strip() else img.get('svg_search_prompt', 'concept')
                target_p = os.path.join(ASSETS_DIR, f"sentence_{int(scene_id):03d}_img_{img_idx+1:02d}.svg")
                slug_n = clean_slug(kw)
                drive_p = os.path.join(drive_gen_path, f"{slug_n}_{random.randint(100,999)}.svg")
                generate_doodle_svg(kw, target_p, drive_p)
                img['svg_search_prompt'] = kw
                img['source'] = f"✨ AI Vẽ Lại: {os.path.basename(drive_p)}"
                
    if found:
        with open("scene_metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        return f"✅ Đã vẽ lại thành công ảnh cho Cảnh #{scene_id} với từ khóa '{custom_kw}'!", generate_preview_cards(meta)
    return f"❌ Không tìm thấy cảnh #{scene_id}", None

# --- THAY THẾ ẢNH TRỰC TIẾP TỪ FILE UPLOAD ---
def replace_scene_image_directly(scene_id, img_index, uploaded_file_obj, drive_gen_path):
    if not os.path.exists("scene_metadata.json"): return "Chưa có dữ liệu kịch bản! Hãy chạy Bước 2 trước.", None
    if not uploaded_file_obj: return "❌ Vui lòng chọn/kéo thả file ảnh cần thay thế!", None
    
    file_path = uploaded_file_obj if isinstance(uploaded_file_obj, str) else uploaded_file_obj.name
    if not file_path or not os.path.exists(file_path):
        return "❌ File tải lên không tồn tại!", None
        
    with open("scene_metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
        
    found = False
    sc_target = int(scene_id)
    idx_target = int(img_index) - 1 # 0-indexed
    
    for s in meta:
        if s.get('sentence_id') == sc_target:
            imgs = s.get('images', [])
            if 0 <= idx_target < len(imgs):
                found = True
                target_svg = os.path.join(ASSETS_DIR, f"sentence_{sc_target:03d}_img_{idx_target+1:02d}.svg")
                ensure_svg_file(file_path, target_svg)
                
                # Lưu bản sao vào Google Drive /MyDrive/image/f/gen/
                os.makedirs(drive_gen_path, exist_ok=True)
                orig_name = os.path.splitext(os.path.basename(file_path))[0]
                drive_p = os.path.join(drive_gen_path, f"upload_c{sc_target}_i{idx_target+1}_{clean_slug(orig_name)}.svg")
                try:
                    shutil.copy(target_svg, drive_p)
                except Exception: pass
                
                imgs[idx_target]['file_name'] = os.path.basename(target_svg)
                imgs[idx_target]['source'] = f"📤 Upload Trực Tiếp: {os.path.basename(file_path)}"
                
    if found:
        with open("scene_metadata.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)
        return f"🎉 Đã thay thế thành công ảnh #{img_index} của Cảnh #{scene_id} bằng file '{os.path.basename(file_path)}'!", generate_preview_cards(meta)
    return f"❌ Không tìm thấy ảnh #{img_index} trong Cảnh #{scene_id}", None

# --- GIAO DIỆN WEB GRADIO HIỆN ĐẠI ---
with gr.Blocks(title="Auto-Scribe V2 Control Center", theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate")) as app:
    gr.Markdown("# 🚀 AUTO-SCRIBE V2: TRUNG TÂM ĐIỀU KHIỂN & CẦU NỐI AI AGENT")
    gr.Markdown("Quy trình 3 bước: Bóc tách gửi Laptop ➔ Kéo kịch bản & Chọn 3 chế độ lấy ảnh ➔ Đóng gói VideoScribe.")
    
    with gr.Tabs():
        with gr.TabItem("🎬 Quy Trình 3 Bước (Xem Trước & Xuất Video)"):
            with gr.Row():
                with gr.Column(scale=1):
                    audio_in = gr.File(label="🎙️ File Giọng Đọc (voiceover.mp3)", file_types=[".mp3", ".wav"])
                    bridge_url_in = gr.Textbox(label="🔗 Local Bridge URL (Cloudflare Tunnel từ Laptop)", placeholder="https://xxxx.trycloudflare.com")
                    
                    image_mode_in = gr.Radio(
                        choices=[
                            "🚀 1. Lấy Ảnh Google Flow từ Laptop qua Tunnel (image-temp/ ➔ Lưu Drive f/gen)",
                            "📂 2. Tìm trong kho Google Drive (/image/f)",
                            "🎨 3. Tự động sinh ảnh AI Doodle 100% (Pollinations + vtracer)"
                        ],
                        value="🚀 1. Lấy Ảnh Google Flow từ Laptop qua Tunnel (image-temp/ ➔ Lưu Drive f/gen)",
                        label="⚙️ 3 Chế Độ Lấy Ảnh"
                    )
                    
                    with gr.Accordion("📂 Cấu hình thư mục Drive (Tùy chọn)", open=False):
                        drive_f_in = gr.Textbox(label="Thư mục ảnh gốc trên Drive", value="/content/drive/MyDrive/image/f")
                        drive_gen_in = gr.Textbox(label="Thư mục lưu ảnh AI sinh mới trên Drive", value="/content/drive/MyDrive/image/f/gen")
                    
                    gr.Markdown("---")
                    btn_step1 = gr.Button("📤 BƯỚC 1: Whisper Bóc Tách & Gửi Về Laptop", variant="primary")
                    btn_step2 = gr.Button("📥 BƯỚC 2: Kéo Kịch Bản & Tải/Vẽ Ảnh (Xem Trước)", variant="secondary")
                    
                    gr.Markdown("---")
                    btn_step3 = gr.Button("📦 BƯỚC 3: ĐÃ ƯNG Ý ➔ BẤM ĐÓNG GÓI & TẢI FILE SCRIBE", variant="primary", size="lg")
                
                with gr.Column(scale=1):
                    log_box = gr.Textbox(label="📋 Nhật Ký Hoạt Động (Live Console Logs)", lines=13, interactive=False)
                    out_file = gr.File(label="📦 Tải File Dự Án VideoScribe (.scribe)")
                    
            gr.Markdown("## 🖼️ Bảng Xem Trước Ảnh Từng Cảnh (Visual Preview)")
            preview_display = gr.HTML(label="Bảng Xem Trước Ảnh")
            
            with gr.Row():
                with gr.Column(scale=1):
                    gr.Markdown("### 📤 Thay Ảnh Trực Tiếp Bằng File Tải Lên")
                    with gr.Row():
                        upl_scene_id = gr.Number(label="Cảnh #", value=1, precision=0, scale=1)
                        upl_img_idx = gr.Number(label="Ảnh #", value=1, precision=0, scale=1)
                    upload_img_in = gr.File(label="Chọn file ảnh thay thế (PNG, JPG, SVG)", file_types=[".png", ".jpg", ".jpeg", ".svg", ".webp"])
                    btn_upload_replace = gr.Button("📤 Tải Lên & Thay Thế Ảnh Cảnh Này", variant="primary")
                    upload_msg = gr.Textbox(label="Thông Báo Thay Ảnh", interactive=False)
                    
                with gr.Column(scale=1):
                    gr.Markdown("### 🎨 Hoặc Yêu Cầu AI Vẽ Lại Bằng Từ Khóa")
                    scene_select = gr.Number(label="Cảnh Muốn Vẽ Lại (Ví dụ: 1)", value=1, precision=0)
                    custom_kw_in = gr.Textbox(label="Từ Khóa Mới Cho Cảnh Này (Tiếng Anh)", placeholder="luxury gold watch, flying rocket...")
                    btn_redraw = gr.Button("🎨 AI Vẽ Lại Ảnh Cho Cảnh Này & Lưu Drive", variant="secondary")
                    redraw_msg = gr.Textbox(label="Thông Báo Vẽ Lại", interactive=False)
            
    btn_step1.click(
        fn=step1_whisper_and_send_to_laptop,
        inputs=[audio_in, bridge_url_in],
        outputs=[log_box]
    )
    
    btn_step2.click(
        fn=step2_pull_from_laptop_and_preview,
        inputs=[bridge_url_in, image_mode_in, drive_f_in, drive_gen_in],
        outputs=[log_box, preview_display]
    )
    
    btn_step3.click(
        fn=step3_build_and_download_scribe,
        inputs=[audio_in],
        outputs=[log_box, out_file]
    )
    
    btn_upload_replace.click(
        fn=replace_scene_image_directly,
        inputs=[upl_scene_id, upl_img_idx, upload_img_in, drive_gen_in],
        outputs=[upload_msg, preview_display]
    )
    
    btn_redraw.click(
        fn=redraw_single_scene,
        inputs=[scene_select, custom_kw_in, drive_gen_in],
        outputs=[redraw_msg, preview_display]
    )

print("🌐 Đang khởi chạy Giao Diện Web & Mở Đường Link Tunnel...")
app.queue().launch(share=True, debug=False, show_error=True)
